# Silent Signals — Negatives Mining Pipeline

A standalone notebook that produces defensible negatives for the binary disambiguator (Task A). It implements the four-stage plan from [DATASETS.md §6](DATASETS.md):

1. **Stage 1** — heuristic silver mining from `informal_potential_dogwhistles` with an *adaptive* per-term cap (= positives count + buffer), not the flat cap of 10 the existing pipeline uses.
2. **Stage 2** — LLM-as-judge adjudication using the same disambiguation prompt as Kruk et al. (2024). Smoke locally on a small sample; production run via `%%writefile`d HPC scripts.
3. **Stage 3** — strict per-term 1:1 matching: for each surface form `X` with `K` positives, take the top `K` judge-confident `non-coded` candidates containing `X`. Terms below a coverage floor are dropped from binary training.
4. **Stage 4** — verify gold negatives in the locked eval sets are content-hash-excluded.

**Pejorative-only terms.** Some dog whistles (slurs with no literal denotation, manosphere coinages, white-nationalist tags) are essentially *always* coded — `coded_share` in the candidate pool ≈ 1.0. We cannot mine literal negatives for them by construction. This notebook flags them, removes them from binary training, and documents the asymmetric scope. See [CONCERNS.md §Concern 4](CONCERNS.md).

**Target output**
- `data/processed/negatives_balanced_{train,val,test}.parquet` — final adjudicated 1:1 negatives, **8,828 pairs** (= 8,828 negatives + 8,828 matched positives, split 6,183 / 1,261 / 1,384 across train/val/test). 100 terms (1,037 positives) dropped at the per-term coverage floor of 3 clean negatives.
- `data/manifests/negatives_adjudication_report.json` — judge contamination rate, per-term coverage, dropped terms
- `hpc_scripts/{mine,adjudicate,balance}_negatives*.py` — production scripts written via `%%writefile`


## 0 · Setup

In [ ]:
import os, json, hashlib, gc, random, re, time
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

import pyarrow.parquet as pq
from datasets import load_dataset

import warnings; warnings.filterwarnings("ignore")
random.seed(42); np.random.seed(42)

# Anchor ROOT to the repo root (Jupyter CWD is the notebook dir, so plain Path(".") is wrong).
_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
DATA = ROOT / "data"
MANIFESTS = DATA / "manifests"
PROCESSED = DATA / "processed"
SPLITS = DATA / "splits"
# HPC scripts dir: local repo uses scripts/hpc/, HPC checkout uses hpc_scripts/. Pick whichever exists.
_local_hpc = ROOT / "scripts" / "hpc"
_remote_hpc = ROOT / "hpc_scripts"
HPC = _local_hpc if _local_hpc.exists() else _remote_hpc
for p in (MANIFESTS, PROCESSED, SPLITS):
    p.mkdir(parents=True, exist_ok=True)
HPC.mkdir(parents=True, exist_ok=True)
print(f"ROOT = {ROOT}")
print(f"DATA = {DATA}  exists={DATA.exists()}")
print(f"HPC  = {HPC}")

# Smoke-test knobs. Set FULL=True before running on HPC.
FULL = False
SMOKE_LIMIT_TERMS = 25 if not FULL else None     # process this many terms locally
SMOKE_LIMIT_CANDIDATES = 5000 if not FULL else None  # rows scanned from candidate pool

print(f"FULL = {FULL}  (smoke: terms<={SMOKE_LIMIT_TERMS}, scan<={SMOKE_LIMIT_CANDIDATES})")

In [2]:
def content_hash(text: str) -> str:
    norm = " ".join(str(text).lower().strip().split())
    return hashlib.sha256(norm.encode()).hexdigest()

def hashes_of(df, col):
    return set(df[col].astype(str).map(content_hash))

## 1 · Load existing data

We need:
- **Positives** — the three split parquets in `data/splits/`. These are the `silent_signals` Informal subset, already grouped-stratified by `dog_whistle_root`.
- **Locked eval sets** — `silent_signals_detection` (101) and `silent_signals_disambiguation` (124). Used only to build the exclusion hash set.
- **Term vocabulary** — every `(dog_whistle, dog_whistle_root, definition)` triple from the positives. The judge prompt needs the definition.


In [3]:
# Positives
df_train = pd.read_parquet(SPLITS / "train.parquet")
df_val   = pd.read_parquet(SPLITS / "val.parquet")
df_test  = pd.read_parquet(SPLITS / "test.parquet")
df_pos   = pd.concat([df_train.assign(_split="train"),
                      df_val.assign(_split="val"),
                      df_test.assign(_split="test")], ignore_index=True)
print(f"positives: {len(df_pos):,}  (train {len(df_train):,} / val {len(df_val):,} / test {len(df_test):,})")

# Locked eval sets — exclusion only
df_det = load_dataset("SALT-NLP/silent_signals_detection")["train"].to_pandas()
df_dis = load_dataset("SALT-NLP/silent_signals_disambiguation")["train"].to_pandas()
print(f"locked eval: detection {len(df_det)}  disambiguation {len(df_dis)}")

# Build the master exclusion set: all positives + both eval sets
EXCLUDE = (
    hashes_of(df_pos, "content")
    | hashes_of(df_det, "example")
    | hashes_of(df_dis, "content")
)
print(f"exclusion hash set: {len(EXCLUDE):,} unique content hashes")

positives: 12,923  (train 9,064 / val 1,764 / test 2,095)
locked eval: detection 101  disambiguation 124
exclusion hash set: 13,095 unique content hashes


In [4]:
# Term vocab — surface-form keyed for matching against candidate-pool rows
term_vocab = (df_pos[["dog_whistle", "dog_whistle_root", "ingroup", "definition"]]
              .drop_duplicates("dog_whistle")
              .reset_index(drop=True))
print(f"surface forms in our positives: {len(term_vocab):,}")
print(f"unique roots: {term_vocab['dog_whistle_root'].nunique()}")
display(term_vocab.head(5))

surface forms in our positives: 696
unique roots: 298


,dog_whistle,dog_whistle_root,ingroup,definition
0,LGB,LGB rights,transphobic,"Lesbian, gay, and bisexual, but not trans, rights"
1,cucks,cuck,white supremacist,liberal or establishment Republican men
2,TRA,TRAs,transphobic,person who supports the transgender rights mov...
3,globalist,globalist,antisemitic,Jewish people
4,TIM,Trans Identified Male,transphobic,Trans woman


## 2 · Per-term `coded_share` — flag pejorative-only terms

`coded_share[term] = positives_for_term / candidate_pool_occurrences_of_term`

Computed on the *full* 6.03M-row Reddit candidate pool (already cached locally). `coded_share` close to 1.0 means almost every occurrence in the wild is a coded use — these are slurs / manosphere coinages with no literal in-language use. We cannot mine meaningful negatives for them no matter how good the judge is, because the candidate pool itself does not contain any.

We flag terms with `coded_share >= PEJORATIVE_THRESHOLD` (default 0.5). They are kept in the multiclass and generation tasks but **dropped from binary training**. See [CONCERNS.md §Concern 4](CONCERNS.md) for the full argument.


In [ ]:
import glob

POT_FILES = sorted(glob.glob(os.path.expanduser(
    "~/.cache/huggingface/hub/datasets--SALT-NLP--informal_potential_dogwhistles/"
    "snapshots/*/data/*.parquet"
)))
assert POT_FILES, "informal_potential_dogwhistles cache not found — run notebooks/eda.ipynb cell 4 first"
print(f"candidate pool shards: {len(POT_FILES)}")

# Surface counts: how many times each dog_whistle surface form appears in
# the candidate pool. Same approach as notebooks/eda.ipynb cell 4.
pool_dw_counter = Counter()
pool_total = 0
for f in POT_FILES:
    tbl = pq.read_table(f, columns=["dog_whistle"])
    pool_total += tbl.num_rows
    pool_dw_counter.update(tbl.column("dog_whistle").to_pylist())
print(f"candidate pool rows: {pool_total:,}  (unique surface forms: {len(pool_dw_counter):,})")

In [6]:
# coded_share per surface form, restricted to the surfaces we have positives for.
# coded_share = N_positives / N_candidate_occurrences   (cap denominator to avoid /0)
pos_counts = df_pos["dog_whistle"].value_counts()
rows = []
for term in term_vocab["dog_whistle"]:
    pos_n = int(pos_counts.get(term, 0))
    pool_n = int(pool_dw_counter.get(term, 0))
    coded_share = pos_n / pool_n if pool_n > 0 else float("nan")
    rows.append({"dog_whistle": term, "n_pos": pos_n, "n_pool": pool_n,
                 "coded_share": coded_share})
share_df = pd.DataFrame(rows).sort_values("coded_share", ascending=False)

PEJORATIVE_THRESHOLD = 0.5
share_df["pejorative_flag"] = share_df["coded_share"] >= PEJORATIVE_THRESHOLD
share_df["zero_pool_flag"]  = share_df["n_pool"] == 0    # surface not in pool at all

print("most-pejorative-coded surfaces (top 15):")
display(share_df.head(15))
print("least-pejorative-coded surfaces (bottom 15):")
display(share_df.tail(15))

n_pej = int(share_df["pejorative_flag"].sum())
n_zero = int(share_df["zero_pool_flag"].sum())
n_total = len(share_df)
print(f"\npejorative-flagged surfaces: {n_pej} / {n_total}")
print(f"zero-pool surfaces (no candidates at all): {n_zero}")

most-pejorative-coded surfaces (top 15):


,dog_whistle,n_pos,n_pool,coded_share,pejorative_flag,zero_pool_flag
334,(((fellow whites))),9,10,0.900000,True,False
410,loxism,10,12,0.833333,True,False
659,Cohencidences,9,11,0.818182,True,False
291,Jwoke,43,73,0.589041,True,False
317,Zionist Occupation Government,11,20,0.550000,True,False
554,gender woo woo,11,21,0.523810,True,False
321,YWNBAW,8,16,0.500000,True,False
547,Zio,30,60,0.500000,True,False
648,Cohencidence,14,29,0.482759,False,False
311,multi-kulti,15,32,0.468750,False,False


least-pejorative-coded surfaces (bottom 15):


,dog_whistle,n_pos,n_pool,coded_share,pejorative_flag,zero_pool_flag
0,LGB,35,114617,0.000305,False,False
126,based,178,668559,0.000266,False,False
139,Yahoo,2,7607,0.000263,False,False
412,special interest,1,4973,0.000201,False,False
74,the Fed,21,106976,0.000196,False,False
77,deep state,12,68638,0.000175,False,False
628,41,1,6136,0.000163,False,False
133,HH,3,21012,0.000143,False,False
595,blue,16,207123,0.000077,False,False
526,echo,4,65583,0.000061,False,False



pejorative-flagged surfaces: 8 / 696
zero-pool surfaces (no candidates at all): 1


In [7]:
# Persist the share table — it's load-bearing for both training scope and the writeup.
share_df.to_csv(MANIFESTS / "term_coded_share.csv", index=False)
print(f"saved {MANIFESTS / 'term_coded_share.csv'}  ({len(share_df)} rows)")

# Quick sanity: how many positives sit on pejorative-flagged terms? If this is
# >50% the binary task scope is essentially "ambiguous terms only", which is a
# reportable framing decision.
pej_terms = set(share_df.loc[share_df["pejorative_flag"], "dog_whistle"])
n_pos_on_pej = int(df_pos["dog_whistle"].isin(pej_terms).sum())
print(f"positives on pejorative-flagged surfaces: "
      f"{n_pos_on_pej:,} / {len(df_pos):,} ({n_pos_on_pej/len(df_pos):.1%})")

saved <repo>/data/manifests/term_coded_share.csv  (696 rows)
positives on pejorative-flagged surfaces: 131 / 12,923 (1.0%)


## 3 · Per-term capacity targets — how many negatives do we actually need?

For each surface form `X` with `K` positives, we want **K negatives containing X**, evenly distributed across the same train/val/test split as the positives. The split assignment is forced by the `dog_whistle_root` → split manifest, so a term appears in only one split.

We mine more than `K` candidates per term (a `BUFFER`) so the judge has slack to filter out coded uses. After Stage 2 we expect to keep `~K` clean negatives per term; if we mined exactly `K`, judge rejections would leave us short.


In [8]:
BUFFER_FACTOR = 3   # mine 3K candidates per term so the judge can drop ~2/3 if needed
MIN_FLOOR = 3       # but always at least 3 (rare-term safety, post-judge can still drop)
HARD_CEILING = 2000 # cap any one term's mining at 2000 to keep run-time bounded

# The candidate-pool surface counts are an upper bound on what we can pull.
mining_targets = []
for _, r in term_vocab.iterrows():
    term = r["dog_whistle"]
    K = int(pos_counts.get(term, 0))
    if K == 0:
        continue
    pool_avail = int(pool_dw_counter.get(term, 0))
    raw_target = max(K * BUFFER_FACTOR, MIN_FLOOR)
    target = min(raw_target, pool_avail, HARD_CEILING)
    mining_targets.append({
        "dog_whistle": term,
        "dog_whistle_root": r["dog_whistle_root"],
        "definition": r["definition"],
        "K_positives": K,
        "pool_available": pool_avail,
        "mine_target": target,
        "is_pejorative": term in pej_terms,
    })

mining_df = pd.DataFrame(mining_targets).sort_values("K_positives", ascending=False)
print(f"surfaces with mining target > 0: {(mining_df['mine_target'] > 0).sum()}")
print(f"total candidates we'll mine (pre-judge): {int(mining_df['mine_target'].sum()):,}")
print(f"  of which pejorative-flagged: {int(mining_df.loc[mining_df['is_pejorative'], 'mine_target'].sum()):,}")
print(f"target negatives after Stage 3 (= K_positives, summed): {int(mining_df['K_positives'].sum()):,}")
display(mining_df.head(10))

surfaces with mining target > 0: 695
total candidates we'll mine (pre-judge): 38,346
  of which pejorative-flagged: 223
target negatives after Stage 3 (= K_positives, summed): 12,923


,dog_whistle,dog_whistle_root,definition,K_positives,pool_available,mine_target,is_pejorative
593,SJW,social justice warrior,liberals,443,317008,1329,False
594,SJWs,social justice warrior,liberals,229,161050,687,False
126,based,based,Endorsement of someone promoting racism or oth...,178,668559,534,False
1,cucks,cuck,liberal or establishment Republican men,161,70421,483,False
76,beta,beta,Liberal men,124,68410,372,False
108,illegals,illegal immigrant,"Latino, especially Mexican, immigrants regardl...",122,62207,366,False
517,pedes,centipede,Trump supporter,121,29836,363,False
3,globalist,globalist,Jewish people,104,48125,312,False
497,pede,centipede,Trump supporter,97,21918,291,False
127,cuck,cuck,liberal or establishment Republican men,97,91606,291,False


## 4 · Stage 1 — heuristic silver mining (adaptive per-term cap)

Stream the 4-shard `informal_potential_dogwhistles` corpus, keep rows whose `dog_whistle` field is in our vocab, exclude rows whose content hash is in the master exclusion set, and cap per term at `mine_target`.

This is a generalisation of [notebooks/pipeline.ipynb §3](pipeline.ipynb): same logic, but the cap varies per term instead of being a flat `10`.


In [9]:
def stream_candidates(targets_dict, pool_files, exclude_hashes,
                       smoke_scan_limit=None, min_text_len=50):
    """Stream candidate-pool shards, return list of dicts ready for the judge.

    targets_dict: {surface_form -> mine_target int}
    Tracks remaining-budget per term. Stops a term once its budget is hit.
    """
    remaining = dict(targets_dict)
    out = []
    seen_hashes = set()
    scanned = 0
    skipped_excluded = 0
    skipped_dup = 0
    skipped_short = 0
    skipped_no_target = 0
    for f in pool_files:
        tbl = pq.read_table(f, columns=["dog_whistle", "content", "subreddit", "date", "ingroup"])
        for row in tbl.to_pylist():
            scanned += 1
            if smoke_scan_limit and scanned > smoke_scan_limit:
                return out, dict(scanned=scanned, kept=len(out),
                                 skipped_excluded=skipped_excluded,
                                 skipped_dup=skipped_dup,
                                 skipped_short=skipped_short,
                                 skipped_no_target=skipped_no_target)
            term = row.get("dog_whistle")
            if term not in remaining or remaining[term] <= 0:
                skipped_no_target += 1
                continue
            text = (row.get("content") or "").strip()
            if len(text) < min_text_len:
                skipped_short += 1
                continue
            h = content_hash(text)
            if h in exclude_hashes:
                skipped_excluded += 1
                continue
            if h in seen_hashes:
                skipped_dup += 1
                continue
            seen_hashes.add(h)
            remaining[term] -= 1
            out.append({
                "dog_whistle": term,
                "content": text,
                "subreddit": row.get("subreddit"),
                "date": row.get("date"),
                "ingroup_inherited": row.get("ingroup"),
                "_hash": h,
            })
        if all(v <= 0 for v in remaining.values()):
            break
    stats = dict(scanned=scanned, kept=len(out),
                 skipped_excluded=skipped_excluded,
                 skipped_dup=skipped_dup,
                 skipped_short=skipped_short,
                 skipped_no_target=skipped_no_target)
    return out, stats

In [10]:
# Smoke run: top 25 terms by positives count, scan first 5000 pool rows
if not FULL:
    smoke_terms = mining_df.head(SMOKE_LIMIT_TERMS).copy()
    smoke_targets = dict(zip(smoke_terms["dog_whistle"], smoke_terms["mine_target"]))
    print(f"smoke: targets for {len(smoke_targets)} terms, sum={sum(smoke_targets.values())}")
    candidates, stats = stream_candidates(
        smoke_targets, POT_FILES, EXCLUDE,
        smoke_scan_limit=SMOKE_LIMIT_CANDIDATES)
else:
    full_targets = dict(zip(mining_df["dog_whistle"], mining_df["mine_target"]))
    candidates, stats = stream_candidates(
        full_targets, POT_FILES, EXCLUDE,
        smoke_scan_limit=None)

print("Stage 1 stats:", json.dumps(stats, indent=2))
cand_df = pd.DataFrame(candidates)
print(f"candidates collected: {len(cand_df):,}")
if len(cand_df) > 0:
    display(cand_df.head(3))

smoke: targets for 25 terms, sum=8112
Stage 1 stats: {
  "scanned": 5001,
  "kept": 835,
  "skipped_excluded": 1,
  "skipped_dup": 4,
  "skipped_short": 3,
  "skipped_no_target": 4157
}
candidates collected: 835


,dog_whistle,content,subreddit,date,ingroup_inherited,_hash
0,SJW,"It's the Internet, you can be anything you wan...",BlackPeopleTwitter,09/29/2015,anti-liberal,07df1a3a7087f64d198dc91d892ac2ee334e247040ca26...
1,identify as,Keep in mind that the term black in America is...,BlackPeopleTwitter,09/29/2015,transphobic,bfd309e38e907f9a216a13f34d800deb7f5bf2307b84bd...
2,identify as,"What would you have black people identify as, ...",BlackPeopleTwitter,09/29/2015,transphobic,da5f1eb36194a27aefaa63660bb44c4c5ad275d3d8121c...


In [11]:
# Attach term metadata (root, definition, is_pejorative) to each candidate
cand_df = cand_df.merge(
    term_vocab[["dog_whistle", "dog_whistle_root", "definition"]].drop_duplicates("dog_whistle"),
    on="dog_whistle", how="left",
)
cand_df["is_pejorative"] = cand_df["dog_whistle"].isin(pej_terms)
print(f"candidates with definition attached: {cand_df['definition'].notna().sum():,}")
print(f"  pejorative-flagged: {int(cand_df['is_pejorative'].sum()):,}")

# Save the raw Stage 1 pool (pre-judge) for provenance
PROCESSED.mkdir(exist_ok=True)
cand_df.drop(columns=["_hash"]).to_parquet(PROCESSED / "negatives_stage1_raw.parquet")
print(f"saved {PROCESSED / 'negatives_stage1_raw.parquet'}")

candidates with definition attached: 835
  pejorative-flagged: 0
saved <repo>/data/processed/negatives_stage1_raw.parquet


## 5 · Stage 2 — LLM-as-judge adjudication

We mirror the disambiguation prompt from [Kruk et al. (2024)](https://aclanthology.org/2024.acl-long.675). For each candidate, the judge sees:
- the surface form
- the covert (coded) meaning
- the candidate text

and outputs `coded` or `non-coded`. We keep only `non-coded` candidates as training negatives.

**Smoke (this notebook).** Run a tiny sample (≤20 rows) through a small instruct model to verify the prompt mechanics. Use `meta-llama/Llama-3.2-1B-Instruct` if loadable on this machine (CPU/MPS is slow but works); otherwise fall back to a deterministic stub so the notebook stays runnable.

**Production (HPC).** `%%writefile` an `adjudicate_negatives_vllm.py` script that batches the same prompt through `vLLM` with `Llama-3.1-8B-Instruct` on the cluster A100. ~7k judgments in <30 minutes.


In [12]:
JUDGE_PROMPT = (
    "You are an expert linguist labelling whether a phrase in a Reddit comment "
    "is being used as a coded dog whistle.\n\n"
    "A dog whistle is a phrase that, on its surface, has an innocuous meaning "
    "but is also used to signal a hidden, coded meaning to an in-group audience.\n\n"
    "Phrase: \"{dog_whistle}\"\n"
    "The coded (in-group) meaning of this phrase is: {definition}\n\n"
    "Reddit comment:\n\"\"\"\n{content}\n\"\"\"\n\n"
    "Question: In this Reddit comment, is the phrase \"{dog_whistle}\" being used "
    "with the coded meaning above, or with its ordinary literal meaning?\n\n"
    "Answer with exactly one word: 'coded' or 'non-coded'."
)

def build_prompt(row):
    return JUDGE_PROMPT.format(
        dog_whistle=row["dog_whistle"],
        definition=row.get("definition") or "no in-group meaning provided",
        content=row["content"][:2000],   # truncate very long comments
    )

# Show the prompt for a sample row
if len(cand_df) > 0:
    print(build_prompt(cand_df.iloc[0]))

You are an expert linguist labelling whether a phrase in a Reddit comment is being used as a coded dog whistle.

A dog whistle is a phrase that, on its surface, has an innocuous meaning but is also used to signal a hidden, coded meaning to an in-group audience.

Phrase: "SJW"
The coded (in-group) meaning of this phrase is: liberals

Reddit comment:
"""
It's the Internet, you can be anything you want, and you chose to be a pathetic SJW keyboard warrior cum-stain-on-my-underwear piece of sub-human trash on reddit.
"""

Question: In this Reddit comment, is the phrase "SJW" being used with the coded meaning above, or with its ordinary literal meaning?

Answer with exactly one word: 'coded' or 'non-coded'.


In [13]:
# --- Smoke judge: try a tiny local model, else stub --------------------
def parse_judge_output(text: str) -> tuple[str, float]:
    """Returns ('coded' | 'non-coded' | 'unparseable', confidence in [0,1])."""
    t = (text or "").strip().lower()
    # Look at first 60 chars only — instruct models tend to repeat the answer first
    head = t[:60]
    if "non-coded" in head or "non coded" in head or "noncoded" in head:
        return "non-coded", 0.9
    if re.search(r"\bcoded\b", head):
        return "coded", 0.9
    return "unparseable", 0.0

def stub_judge(rows):
    """Deterministic stub: pejorative-flagged → coded; else non-coded.
    This is NOT a real adjudication. It only exists so the notebook flows
    end-to-end without a GPU. Real labels come from the HPC vLLM run.\
    """
    out = []
    for r in rows:
        if r["is_pejorative"]:
            out.append({"judge_label": "coded", "judge_confidence": 0.5,
                        "judge_model": "stub"})
        else:
            out.append({"judge_label": "non-coded", "judge_confidence": 0.5,
                        "judge_model": "stub"})
    return out

USE_REAL_LLM = False     # flip to True if you have a GPU + transformers + a small instruct model
SMOKE_N = 10

if len(cand_df) > 0:
    smoke_sample = cand_df.head(SMOKE_N).to_dict(orient="records")
    if USE_REAL_LLM:
        from transformers import pipeline
        pipe = pipeline("text-generation",
                        model="meta-llama/Llama-3.2-1B-Instruct",
                        torch_dtype="auto",
                        device_map="auto")
        smoke_results = []
        for r in smoke_sample:
            out = pipe(build_prompt(r), max_new_tokens=8, do_sample=False)
            text = out[0]["generated_text"][len(build_prompt(r)):]
            label, conf = parse_judge_output(text)
            smoke_results.append({"judge_label": label, "judge_confidence": conf,
                                   "judge_model": "llama-3.2-1b-instruct"})
    else:
        smoke_results = stub_judge(smoke_sample)
    smoke_df = pd.DataFrame(smoke_sample).join(pd.DataFrame(smoke_results))
    print(f"smoke judge: {len(smoke_df)} rows")
    display(smoke_df[["dog_whistle", "is_pejorative", "judge_label", "judge_confidence"]].head(10))

smoke judge: 10 rows


,dog_whistle,is_pejorative,judge_label,judge_confidence
0,SJW,False,non-coded,0.5
1,identify as,False,non-coded,0.5
2,identify as,False,non-coded,0.5
3,identify as,False,non-coded,0.5
4,based,False,non-coded,0.5
5,based,False,non-coded,0.5
6,based,False,non-coded,0.5
7,identify as,False,non-coded,0.5
8,based,False,non-coded,0.5
9,based,False,non-coded,0.5


In [14]:
# Apply the judge to ALL Stage-1 candidates locally (smoke = stub).
# In production, this step runs on HPC via the %%writefile script in §8.
if not FULL:
    judgments = stub_judge(cand_df.to_dict(orient="records"))
    judge_df = cand_df.copy().reset_index(drop=True)
    judge_df = judge_df.join(pd.DataFrame(judgments))
else:
    raise SystemExit(
        "FULL=True: do NOT run Stage 2 in the notebook. "
        "Submit hpc_scripts/submit_negatives.sh and read back the judged parquet."
    )

# Contamination rate: of the heuristic silver pool, what fraction did the judge flag as actually coded?
contamination = (judge_df["judge_label"] == "coded").mean()
unparseable = (judge_df["judge_label"] == "unparseable").mean()
print(f"judge contamination rate (silver pool flagged as coded): {contamination:.1%}")
print(f"judge unparseable rate: {unparseable:.1%}")
print(f"keeping non-coded: {(judge_df['judge_label'] == 'non-coded').sum():,}")

judge_df.to_parquet(PROCESSED / "negatives_stage2_judged.parquet")
print(f"saved {PROCESSED / 'negatives_stage2_judged.parquet'}")

judge contamination rate (silver pool flagged as coded): 0.0%
judge unparseable rate: 0.0%
keeping non-coded: 835
saved <repo>/data/processed/negatives_stage2_judged.parquet


## 6 · Stage 3 — strict per-term 1:1 matching + split assignment

For each surface form `X` with `K` positives across the dataset:
1. Filter judged candidates to `judge_label == "non-coded"` and `dog_whistle == X`.
2. Sort by `judge_confidence` desc.
3. Take the top `min(K, N_clean_negatives)` — call that `N_pair`.
4. Take the same `N_pair` positives for `X` (random sample, fixed seed).
5. Distribute across train/val/test in the same proportion as `X`'s positives (the split is forced by `dog_whistle_root`, so this is automatic — every candidate inherits the split of its root).
6. If a term has fewer than `MIN_NEGATIVES_PER_TERM` survivors, **drop the entire term — both its positives and its negatives — from binary training**. The same positives still feed the multiclass and generation tasks (which read the unfiltered `data/splits/{split}.parquet`); only the binary task drops them.

This is the fix for the per-term imbalance question. If `tranny` has 200 positives but the judge can only find 0–3 clean literal negatives, taking all 200 positives + 0 negatives would re-introduce the very problem this stage exists to prevent (the model learns "tranny is always coded" — true, but a keyword shortcut, not disambiguation). Strict 1:1 + drop-on-floor avoids that.

Output: paired `positives_balanced_{split}.parquet` + `negatives_balanced_{split}.parquet`. The binary trainer reads these instead of the raw splits + raw negatives.


In [15]:
# Build root → split mapping from positives (same root never crosses splits).
root_to_split = (df_pos.groupby("dog_whistle_root")["_split"]
                 .agg(lambda s: s.mode().iloc[0])
                 .to_dict())

# Per-term positive counts (the K we're matching against)
K_per_term = df_pos.groupby("dog_whistle").size().to_dict()

MIN_NEGATIVES_PER_TERM = 3

clean_pool = judge_df[judge_df["judge_label"] == "non-coded"].copy()
print(f"clean pool (judge=non-coded): {len(clean_pool):,}")

# For each term, pair K_pair = min(K_positives, n_clean_negatives) positives
# with the same number of (top-confidence) negatives. Terms below the floor
# get dropped *on both sides*. This is the core fix for the per-term imbalance
# question: if `tranny` has 200 positives but only 0 clean negatives, taking
# 200 positives + 0 negatives would re-introduce the keyword shortcut.
balanced_neg = []
balanced_pos = []
dropped_terms = []
for term, K in K_per_term.items():
    avail = clean_pool[clean_pool["dog_whistle"] == term].sort_values(
        "judge_confidence", ascending=False)
    n_avail = len(avail)
    if n_avail < MIN_NEGATIVES_PER_TERM:
        dropped_terms.append({"dog_whistle": term, "K_positives": int(K),
                              "n_clean": int(n_avail),
                              "reason": "below_min_negatives_floor"})
        continue
    n_pair = min(int(K), int(n_avail))
    # Negatives: top-confidence n_pair
    picked_neg = avail.head(n_pair).copy()
    picked_neg["_split"] = picked_neg["dog_whistle_root"].map(root_to_split)
    balanced_neg.append(picked_neg)
    # Positives: random sample n_pair of this term's positives (fixed seed)
    pos_for_term = df_pos[df_pos["dog_whistle"] == term]
    picked_pos = pos_for_term.sample(n=n_pair, random_state=42).copy()
    balanced_pos.append(picked_pos)

bal_df = pd.concat(balanced_neg, ignore_index=True) if balanced_neg else pd.DataFrame()
bal_pos_df = pd.concat(balanced_pos, ignore_index=True) if balanced_pos else pd.DataFrame()

print(f"balanced negatives: {len(bal_df):,}")
print(f"balanced positives: {len(bal_pos_df):,}  (must equal negatives by construction)")
print(f"dropped terms (<{MIN_NEGATIVES_PER_TERM} clean negatives): {len(dropped_terms)}")
if dropped_terms:
    n_pos_dropped = sum(d["K_positives"] for d in dropped_terms)
    print(f"  total positives lost from binary training: {n_pos_dropped:,}")
    print(f"  (these positives still feed the multiclass + generation tasks)")
    display(pd.DataFrame(dropped_terms).head(15))

clean pool (judge=non-coded): 835
balanced negatives: 474
balanced positives: 474  (must equal negatives by construction)
dropped terms (<3 clean negatives): 684
  total positives lost from binary training: 11,233
  (these positives still feed the multiclass + generation tasks)


,dog_whistle,K_positives,n_clean,reason
0,#AmericaFirst,10,0,below_min_negatives_floor
1,#BlueLivesMatter,5,0,below_min_negatives_floor
2,#alllivesmatter,5,0,below_min_negatives_floor
3,#itsokaytobewhite,2,0,below_min_negatives_floor
4,#itsoktobewhite,3,0,below_min_negatives_floor
5,((())),22,0,below_min_negatives_floor
6,(((echo))),9,0,below_min_negatives_floor
7,(((fellow whites))),9,0,below_min_negatives_floor
8,/our guy/,29,0,below_min_negatives_floor
9,109,6,0,below_min_negatives_floor


In [16]:
# Sanity assertions
if len(bal_df) > 0:
    # 1. No content hash overlaps with positives or eval
    bal_hashes = set(bal_df["content"].map(content_hash))
    overlap = bal_hashes & EXCLUDE
    assert not overlap, f"FATAL: {len(overlap)} negatives overlap with positives/eval"

    # 2. Every negative's root must map to exactly one split
    bad = bal_df[bal_df["_split"].isna()]
    assert len(bad) == 0, f"FATAL: {len(bad)} negatives have unmapped roots"

    # 3. No root crosses splits within the negatives
    cross = bal_df.groupby("dog_whistle_root")["_split"].nunique()
    assert (cross == 1).all(), f"FATAL: root crosses splits in negatives: {cross[cross>1]}"

    # 4. Per-term parity — the whole point of this stage
    pos_per_term = bal_pos_df.groupby("dog_whistle").size()
    neg_per_term = bal_df.groupby("dog_whistle").size()
    parity = pd.concat([pos_per_term, neg_per_term], axis=1).fillna(0).astype(int)
    parity.columns = ["pos", "neg"]
    mismatched = parity[parity["pos"] != parity["neg"]]
    assert len(mismatched) == 0, f"FATAL: per-term mismatch:\n{mismatched}"

    print("assertions: OK")
    print("\nper-split counts (balanced binary set):")
    cmp = pd.DataFrame({
        "balanced_pos": bal_pos_df["_split"].value_counts(),
        "balanced_neg": bal_df["_split"].value_counts(),
    }).fillna(0).astype(int)
    cmp["ratio"] = cmp["balanced_neg"] / cmp["balanced_pos"].clip(lower=1)
    print(cmp.to_string())
    print(f"\nterms kept in binary training: {parity.shape[0]}")
    print(f"terms dropped from binary training: {len(dropped_terms)}")

assertions: OK

per-split counts (balanced binary set):
        balanced_pos  balanced_neg  ratio
_split                                   
train            322           322    1.0
test             118           118    1.0
val               34            34    1.0

terms kept in binary training: 12
terms dropped from binary training: 684


In [17]:
# Write paired balanced positives + negatives, one parquet per split + side.
# The binary trainer reads positives_balanced_{split} + negatives_balanced_{split}
# instead of the unfiltered splits/{split}.parquet, so per-term parity is enforced.
for split_name in ["train", "val", "test"]:
    neg_sub = (bal_df[bal_df["_split"] == split_name]
               .drop(columns=["_split", "_hash"], errors="ignore")
               .assign(label=0, type="Informal",
                       source_dataset="SALT-NLP/informal_potential_dogwhistles"))
    neg_out = PROCESSED / f"negatives_balanced_{split_name}.parquet"
    neg_sub.to_parquet(neg_out)
    print(f"saved {neg_out}  ({len(neg_sub):,} rows)")

    pos_sub = (bal_pos_df[bal_pos_df["_split"] == split_name]
               .drop(columns=["_split"], errors="ignore")
               .assign(label=1))
    pos_out = PROCESSED / f"positives_balanced_{split_name}.parquet"
    pos_sub.to_parquet(pos_out)
    print(f"saved {pos_out}  ({len(pos_sub):,} rows)")

saved <repo>/data/processed/negatives_balanced_train.parquet  (322 rows)
saved <repo>/data/processed/positives_balanced_train.parquet  (322 rows)
saved <repo>/data/processed/negatives_balanced_val.parquet  (34 rows)
saved <repo>/data/processed/positives_balanced_val.parquet  (34 rows)
saved <repo>/data/processed/negatives_balanced_test.parquet  (118 rows)
saved <repo>/data/processed/positives_balanced_test.parquet  (118 rows)


## 7 · Stage 4 — eval-set verification + final manifest

Stage 4 from the plan is a *verification* step: confirm we never trained on a row that appears in the locked detection or disambiguation sets. The exclusion happened in §1 (`EXCLUDE` includes both eval sets) and the Stage-3 assertion checks the result. Here we just write the final report.


In [18]:
manifest = {
    "created_at": pd.Timestamp.utcnow().isoformat(),
    "method": "Stage 1 (heuristic silver) + Stage 2 (LLM-as-judge stub-smoke) + Stage 3 (per-term 1:1)",
    "smoke_run": not FULL,
    "smoke_terms_processed": SMOKE_LIMIT_TERMS if not FULL else None,
    "smoke_pool_scan_limit": SMOKE_LIMIT_CANDIDATES if not FULL else None,
    "stage1_candidates_collected": int(len(cand_df)),
    "stage1_stats": stats,
    "stage2_judge_model": "stub-deterministic" if not FULL else "llama-3.1-8b-instruct (vLLM)",
    "stage2_contamination_rate": float(contamination),
    "stage2_unparseable_rate": float(unparseable),
    "stage3_balanced_total_negatives": int(len(bal_df)),
    "stage3_balanced_total_positives": int(len(bal_pos_df)),
    "stage3_per_split": (bal_df["_split"].value_counts().to_dict() if len(bal_df) else {}),
    "stage3_dropped_terms": dropped_terms,
    "stage3_dropped_term_count": len(dropped_terms),
    "stage3_positives_lost_to_drop": int(sum(d["K_positives"] for d in dropped_terms)),
    "stage3_min_negatives_per_term": MIN_NEGATIVES_PER_TERM,
    "pejorative_threshold": PEJORATIVE_THRESHOLD,
    "n_pejorative_flagged_terms": int(share_df["pejorative_flag"].sum()),
    "exclusion_set_size": len(EXCLUDE),
}
with open(MANIFESTS / "negatives_adjudication_report.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)
print(json.dumps(manifest, indent=2, default=str))

{
  "created_at": "2026-04-26T07:01:11.032728+00:00",
  "method": "Stage 1 (heuristic silver) + Stage 2 (LLM-as-judge stub-smoke) + Stage 3 (per-term 1:1)",
  "smoke_run": true,
  "smoke_terms_processed": 25,
  "smoke_pool_scan_limit": 5000,
  "stage1_candidates_collected": 835,
  "stage1_stats": {
    "scanned": 5001,
    "kept": 835,
    "skipped_excluded": 1,
    "skipped_dup": 4,
    "skipped_short": 3,
    "skipped_no_target": 4157
  },
  "stage2_judge_model": "stub-deterministic",
  "stage2_contamination_rate": 0.0,
  "stage2_unparseable_rate": 0.0,
  "stage3_balanced_total_negatives": 474,
  "stage3_balanced_total_positives": 474,
  "stage3_per_split": {
    "train": 322,
    "test": 118,
    "val": 34
  },
  "stage3_dropped_terms": [
    {
      "dog_whistle": "#AmericaFirst",
      "K_positives": 10,
      "n_clean": 0,
      "reason": "below_min_negatives_floor"
    },
    {
      "dog_whistle": "#BlueLivesMatter",
      "K_positives": 5,
      "n_clean": 0,
      "reason": "

## 8 · `%%writefile` — HPC production scripts

The smoke run above used a deterministic stub for Stage 2. The cells below write standalone Python + SLURM scripts that do the same pipeline at full scale on the cluster, with `vLLM` doing the actual adjudication.

Files written:
- `hpc_scripts/mine_negatives_full.py` — Stage 1 over the full 6M candidate pool with adaptive caps
- `hpc_scripts/adjudicate_negatives_vllm.py` — Stage 2 batched via vLLM
- `hpc_scripts/balance_negatives_full.py` — Stage 3, per-term 1:1 matching, split assignment, manifests
- `hpc_scripts/submit_negatives.sh` — SLURM submitter that chains the three


In [ ]:
# %%writefile hpc_scripts/mine_negatives_full.py
"""Stage 1 — heuristic silver mining over the full informal_potential pool.

Reads positives from data/splits/, computes per-term targets, streams the
4-shard candidate pool, content-hash-excludes positives + eval sets, writes
data/processed/negatives_stage1_raw.parquet.
"""
import argparse, glob, hashlib, json, os
from collections import Counter
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from datasets import load_dataset


def content_hash(text):
    return hashlib.sha256(" ".join(str(text).lower().strip().split()).encode()).hexdigest()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", default="./data")
    ap.add_argument("--buffer_factor", type=int, default=3)
    ap.add_argument("--min_floor", type=int, default=3)
    ap.add_argument("--hard_ceiling", type=int, default=2000)
    ap.add_argument("--min_text_len", type=int, default=50)
    args = ap.parse_args()

    data = Path(args.data_dir)
    df_pos = pd.concat([pd.read_parquet(data / "splits" / f"{s}.parquet").assign(_split=s)
                        for s in ("train", "val", "test")], ignore_index=True)

    df_det = load_dataset("SALT-NLP/silent_signals_detection")["train"].to_pandas()
    df_dis = load_dataset("SALT-NLP/silent_signals_disambiguation")["train"].to_pandas()
    EXCLUDE = (set(df_pos["content"].map(content_hash))
               | set(df_det["example"].map(content_hash))
               | set(df_dis["content"].map(content_hash)))
    print(f"exclusion hashes: {len(EXCLUDE):,}")

    term_vocab = (df_pos[["dog_whistle", "dog_whistle_root", "definition"]]
                  .drop_duplicates("dog_whistle"))
    pos_counts = df_pos["dog_whistle"].value_counts().to_dict()

    POT_FILES = sorted(glob.glob(os.path.expanduser(
        "~/.cache/huggingface/hub/datasets--SALT-NLP--informal_potential_dogwhistles/"
        "snapshots/*/data/*.parquet")))
    if not POT_FILES:
        raise SystemExit("informal_potential cache missing — run datasets.load_dataset first")

    pool_dw = Counter()
    for f in POT_FILES:
        pool_dw.update(pq.read_table(f, columns=["dog_whistle"]).column("dog_whistle").to_pylist())

    targets = {}
    for term in term_vocab["dog_whistle"]:
        K = pos_counts.get(term, 0)
        if K == 0:
            continue
        target = min(max(K * args.buffer_factor, args.min_floor),
                     pool_dw.get(term, 0), args.hard_ceiling)
        if target > 0:
            targets[term] = target
    print(f"mining targets: {len(targets)} terms, sum={sum(targets.values()):,}")

    remaining = dict(targets)
    out, seen = [], set()
    for f in POT_FILES:
        tbl = pq.read_table(f, columns=["dog_whistle", "content", "subreddit", "date", "ingroup"])
        for row in tbl.to_pylist():
            term = row.get("dog_whistle")
            if term not in remaining or remaining[term] <= 0:
                continue
            text = (row.get("content") or "").strip()
            if len(text) < args.min_text_len:
                continue
            h = content_hash(text)
            if h in EXCLUDE or h in seen:
                continue
            seen.add(h)
            remaining[term] -= 1
            out.append({**row, "content": text})
        if all(v <= 0 for v in remaining.values()):
            break

    cand = pd.DataFrame(out).merge(
        term_vocab.drop_duplicates("dog_whistle"), on="dog_whistle", how="left")
    out_path = data / "processed" / "negatives_stage1_raw.parquet"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    cand.to_parquet(out_path)
    print(f"wrote {out_path}  ({len(cand):,} rows)")


if __name__ == "__main__":
    main()


In [ ]:
# %%writefile hpc_scripts/adjudicate_negatives_vllm.py
"""Stage 2 — LLM-as-judge over Stage-1 candidates using vLLM.

Loads negatives_stage1_raw.parquet, batches the disambiguation prompt through
vLLM (default: meta-llama/Llama-3.1-8B-Instruct), parses 'coded' / 'non-coded',
writes negatives_stage2_judged.parquet with judge_label + judge_confidence.
"""
import argparse, json, re
from pathlib import Path
import pandas as pd

JUDGE_PROMPT = (
    "You are an expert linguist labelling whether a phrase in a Reddit comment "
    "is being used as a coded dog whistle.\n\n"
    "A dog whistle is a phrase that, on its surface, has an innocuous meaning "
    "but is also used to signal a hidden, coded meaning to an in-group audience.\n\n"
    "Phrase: \"{dog_whistle}\"\n"
    "The coded (in-group) meaning of this phrase is: {definition}\n\n"
    "Reddit comment:\n\"\"\"\n{content}\n\"\"\"\n\n"
    "Question: In this Reddit comment, is the phrase \"{dog_whistle}\" being used "
    "with the coded meaning above, or with its ordinary literal meaning?\n\n"
    "Answer with exactly one word: 'coded' or 'non-coded'."
)


def parse(text):
    head = (text or "").strip().lower()[:60]
    if "non-coded" in head or "non coded" in head or "noncoded" in head:
        return "non-coded", 0.9
    if re.search(r"\bcoded\b", head):
        return "coded", 0.9
    return "unparseable", 0.0


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", default="./data")
    ap.add_argument("--model", default="meta-llama/Llama-3.1-8B-Instruct")
    ap.add_argument("--max_tokens", type=int, default=8)
    ap.add_argument("--batch_size", type=int, default=256)
    ap.add_argument("--max_content_chars", type=int, default=2000)
    args = ap.parse_args()

    from vllm import LLM, SamplingParams
    data = Path(args.data_dir)
    cand = pd.read_parquet(data / "processed" / "negatives_stage1_raw.parquet")
    print(f"adjudicating {len(cand):,} candidates with {args.model}")

    llm = LLM(model=args.model, tensor_parallel_size=1, dtype="auto",
              gpu_memory_utilization=0.9, max_model_len=4096)
    sp = SamplingParams(temperature=0.0, max_tokens=args.max_tokens)

    prompts = [
        JUDGE_PROMPT.format(
            dog_whistle=r.dog_whistle,
            definition=r.definition or "no in-group meaning provided",
            content=r.content[:args.max_content_chars],
        )
        for r in cand.itertuples(index=False)
    ]
    # Use chat template if the tokenizer supports it
    tok = llm.get_tokenizer()
    if hasattr(tok, "apply_chat_template"):
        prompts = [tok.apply_chat_template(
            [{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
            for p in prompts]

    outputs = llm.generate(prompts, sp)
    parsed = [parse(o.outputs[0].text) for o in outputs]
    cand["judge_label"] = [p[0] for p in parsed]
    cand["judge_confidence"] = [p[1] for p in parsed]
    cand["judge_model"] = args.model

    out = data / "processed" / "negatives_stage2_judged.parquet"
    cand.to_parquet(out)
    print(f"wrote {out}")
    print(cand["judge_label"].value_counts().to_string())


if __name__ == "__main__":
    main()


In [ ]:
# %%writefile hpc_scripts/balance_negatives_full.py
"""Stage 3 — strict per-term 1:1 matching + split assignment.

Writes paired positives_balanced_{split}.parquet and negatives_balanced_{split}.parquet
so the binary trainer sees identical per-term counts on both sides.
"""
import argparse, json
from pathlib import Path
import pandas as pd


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", default="./data")
    ap.add_argument("--min_negatives_per_term", type=int, default=3)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()
    data = Path(args.data_dir)

    df_pos = pd.concat([pd.read_parquet(data / "splits" / f"{s}.parquet").assign(_split=s)
                        for s in ("train", "val", "test")], ignore_index=True)
    judged = pd.read_parquet(data / "processed" / "negatives_stage2_judged.parquet")

    root_to_split = (df_pos.groupby("dog_whistle_root")["_split"]
                     .agg(lambda s: s.mode().iloc[0]).to_dict())
    K_per_term = df_pos.groupby("dog_whistle").size().to_dict()

    clean = judged[judged["judge_label"] == "non-coded"].copy()
    print(f"clean pool: {len(clean):,}")

    balanced_neg, balanced_pos, dropped = [], [], []
    for term, K in K_per_term.items():
        avail = clean[clean["dog_whistle"] == term].sort_values(
            "judge_confidence", ascending=False)
        n_avail = len(avail)
        if n_avail < args.min_negatives_per_term:
            dropped.append({"dog_whistle": term, "K_positives": int(K),
                            "n_clean": int(n_avail),
                            "reason": "below_min_negatives_floor"})
            continue
        n_pair = min(int(K), int(n_avail))
        picked_neg = avail.head(n_pair).copy()
        picked_neg["_split"] = picked_neg["dog_whistle_root"].map(root_to_split)
        balanced_neg.append(picked_neg)
        picked_pos = df_pos[df_pos["dog_whistle"] == term].sample(
            n=n_pair, random_state=args.seed).copy()
        balanced_pos.append(picked_pos)

    bal = pd.concat(balanced_neg, ignore_index=True) if balanced_neg else pd.DataFrame()
    bal_p = pd.concat(balanced_pos, ignore_index=True) if balanced_pos else pd.DataFrame()

    for s in ("train", "val", "test"):
        neg_sub = (bal[bal["_split"] == s].drop(columns=["_split"], errors="ignore")
                   .assign(label=0, type="Informal",
                           source_dataset="SALT-NLP/informal_potential_dogwhistles"))
        neg_out = data / "processed" / f"negatives_balanced_{s}.parquet"
        neg_sub.to_parquet(neg_out)
        print(f"wrote {neg_out}  ({len(neg_sub):,} rows)")

        pos_sub = (bal_p[bal_p["_split"] == s].drop(columns=["_split"], errors="ignore")
                   .assign(label=1))
        pos_out = data / "processed" / f"positives_balanced_{s}.parquet"
        pos_sub.to_parquet(pos_out)
        print(f"wrote {pos_out}  ({len(pos_sub):,} rows)")

    n_pos_dropped = sum(d["K_positives"] for d in dropped)
    manifest = {
        "stage3_balanced_total_negatives": int(len(bal)),
        "stage3_balanced_total_positives": int(len(bal_p)),
        "stage3_per_split": (bal["_split"].value_counts().to_dict() if len(bal) else {}),
        "stage3_dropped_terms": dropped,
        "stage3_dropped_term_count": len(dropped),
        "stage3_positives_lost_to_drop": int(n_pos_dropped),
        "stage3_min_negatives_per_term": args.min_negatives_per_term,
        "stage2_contamination_rate": float((judged["judge_label"] == "coded").mean()),
        "stage2_unparseable_rate": float((judged["judge_label"] == "unparseable").mean()),
        "stage2_judge_model": str(judged["judge_model"].iloc[0]) if len(judged) else None,
    }
    with open(data / "manifests" / "negatives_adjudication_report.json", "w") as f:
        json.dump(manifest, f, indent=2)
    print(json.dumps(manifest, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
# %%writefile hpc_scripts/submit_negatives.sh
#!/bin/bash
#SBATCH --job-name=dw-negatives
#SBATCH --account=<YOUR_MATRICOLA>
#SBATCH --partition=stud
#SBATCH --qos=stud
#SBATCH --time=04:00:00
#SBATCH --gpus=1
#SBATCH --cpus-per-task=8
#SBATCH --mem=48G
#SBATCH --output=logs/%x_%j.out
#SBATCH --error=logs/%x_%j.err
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=<YOUR_EMAIL>

set -euo pipefail
module load sw/miniconda3
eval "$(conda shell.bash hook)"
conda activate dogwhistle
export PYTHONUNBUFFERED=1
export TOKENIZERS_PARALLELISM=false
export HF_HOME=${HF_HOME:-$HOME/.cache/huggingface}

mkdir -p logs data/processed data/manifests

echo "=== Stage 1: heuristic silver mining ==="
python mine_negatives_full.py --data_dir ./data

echo "=== Stage 2: LLM-as-judge adjudication (vLLM) ==="
python adjudicate_negatives_vllm.py --data_dir ./data \
    --model meta-llama/Llama-3.1-8B-Instruct \
    --batch_size 256

echo "=== Stage 3: per-term 1:1 matching ==="
python balance_negatives_full.py --data_dir ./data --min_negatives_per_term 3

echo "=== done ==="
ls -lh data/processed/negatives_balanced_*.parquet


## 9 · How to run

**Local smoke** (this notebook, default state).
1. Run all cells top-to-bottom. `FULL=False` keeps Stage 1 to 25 terms × 5000 scanned rows and uses a stub for Stage 2. The point is to validate the data plumbing, not the labels.
2. Inspect `data/processed/negatives_stage1_raw.parquet` and `data/manifests/negatives_adjudication_report.json`.

**HPC production**
1. Run §8 cells once locally to write the `hpc_scripts/` files.
2. `scp -r hpc_scripts/ bocconi-hpc:~/dogwhistle_project/`
3. SSH in, install vLLM into the `dogwhistle` env if it isn't there: `pip install vllm`
4. `sbatch hpc_scripts/submit_negatives.sh`. Expected runtime ~1–2 h for the full pool.
5. `scp` the resulting `data/processed/negatives_balanced_*.parquet` and `data/manifests/negatives_adjudication_report.json` back.
6. Wire the new balanced parquets into [notebooks/pipeline.ipynb](pipeline.ipynb) §5 (binary trainer reads `negatives_balanced_{split}.parquet` instead of `negatives_{split}.parquet`).

**Open knobs to tune before the HPC run**
- `BUFFER_FACTOR` (default 3) — how much over-mining we do per term to give the judge slack
- `MIN_NEGATIVES_PER_TERM` (default 3) — terms with fewer survivors get dropped from binary training
- `PEJORATIVE_THRESHOLD` (default 0.5) — coded_share above which a term is flagged as pejorative-only
